# Notebook 01 — Decay + Carleman Weight Visualization

**Repo:** `unique-continuation-constraint-lab`  
**Role:** generate the first reproducible visual layer for the unique-continuation constraint pipeline.

Pipeline focus:

\[
\text{two-time decay}
\rightarrow
\text{Carleman weight}
\rightarrow
\text{smallness acts under constraints}
\]

**same math · lifted clarity**

This notebook is an expository visualization. It does not prove a theorem. It creates finite-grid figures that match the paper/page language:

- Step 1: two-time decay
- Step 2: exponential / Carleman weighting
- local CGCS alignment for the first two proof steps


## 0. Setup

This version uses repo modules when available:

```text
src/
  weights.py
  plotting.py
```

Generated figures:

```text
figures/
  step1_two_time_decay_code.png
  step2_exponential_weight_code.png
  notebook01_ab_now_alignment_code.png
```


In [ ]:
import sys
from pathlib import Path

# Works from repo root, notebooks/, or Colab.
for candidate in [Path(".."), Path(".")]:
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate))
        break

import numpy as np
import matplotlib.pyplot as plt

try:
    from src.weights import (
        gaussian,
        gaussian_decay_envelope,
        bounded_weight_lens,
        weighted_signal,
        normalize,
    )
    from src.plotting import set_style, ensure_fig_dir, save, add_caption, plot_alignment_bars
except Exception:
    # Colab/repo fallback if src is not yet installed.
    def gaussian(x, center=0.0, width=1.0, amplitude=1.0):
        return amplitude * np.exp(-((x - center) ** 2) / (2 * width ** 2))

    def gaussian_decay_envelope(x, C=1.0, a=0.4):
        return C * np.exp(-a * x**2)

    def bounded_weight_lens(x, tau=0.22, floor=0.45, height=0.70):
        return floor + height * np.exp(-tau * x**2)

    def weighted_signal(u, w):
        return w * u

    def normalize(u):
        m = np.max(np.abs(u))
        return u if m == 0 else u / m

    def set_style():
        plt.rcParams.update({
            "figure.figsize": (10, 4.8),
            "font.size": 12,
            "axes.grid": True,
            "grid.alpha": 0.25,
            "axes.spines.top": False,
            "axes.spines.right": False,
        })

    def ensure_fig_dir(path="figures"):
        candidates = [Path("../figures"), Path(path)]
        for c in candidates:
            if c.exists():
                c.mkdir(parents=True, exist_ok=True)
                return c
        candidates[0].mkdir(parents=True, exist_ok=True)
        return candidates[0]

    def save(fig, path, dpi=180):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=dpi, bbox_inches="tight")

    def add_caption(ax, text, y=-0.22):
        ax.text(
            0.02, y, text,
            transform=ax.transAxes,
            fontsize=11,
            color="0.25",
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.75"),
        )

    def plot_alignment_bars(labels, scores, title, target=0.96):
        gate_45 = 1 / np.sqrt(2)
        fig, ax = plt.subplots(figsize=(10, 4.8))
        bars = ax.bar(labels, scores)
        ax.axhline(target, linestyle="--", linewidth=2, label="max-CGCS target 0.96")
        ax.axhline(gate_45, linestyle=":", linewidth=2, label=r"45° gate $1/\sqrt{1^2+1^2}$")
        for bar, score in zip(bars, scores):
            ax.text(bar.get_x()+bar.get_width()/2, score+0.015, f"{score:.3f}",
                    ha="center", va="bottom", fontsize=11)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("local CGCS")
        ax.set_title(title, fontsize=18, pad=14)
        ax.legend(loc="lower right")
        return fig, ax

set_style()
FIG_DIR = ensure_fig_dir()
print(f"Figure directory: {FIG_DIR}")

## 1. Two-time decay profiles

We model two snapshots of a Schrödinger solution with Gaussian-like profiles.

This is a visualization of the two-time decay assumption:

\[
|u(x,t_1)| \le C e^{-a|x|^2},
\qquad
|u(x,t_2)| \le C e^{-a|x|^2}.
\]

CGCS bridge language:

- **two-time decay** = paired constraint input
- **sister diagonal** = two-time constraint pairing
- **zero solution** remains only after the full proof pipeline, not at this first step


In [ ]:
x = np.linspace(-5, 5, 1200)

u_t1 = gaussian(x, center=-0.15, width=0.52, amplitude=1.00)
u_t2 = gaussian(x, center=0.65, width=0.48, amplitude=0.86)

a = 0.42
C = 1.08
envelope = gaussian_decay_envelope(x, C=C, a=a)

print("Generated two finite-grid profiles u(x,t1), u(x,t2)")
print(f"Decay envelope: C exp(-a x^2), with C={C}, a={a}")

## 2. Figure — Step 1: two-time decay

Paper-ready filename:

```text
figures/step1_two_time_decay_code.png
```

Recommended caption:

> Step 1. Two-time decay: the candidate solution is small at two different sister-diagonal times \(t_1\) and \(t_2\). This paired constraint is the input to the continuation pipeline.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2))

ax.plot(x, u_t1, linewidth=2.5, label=r"$|u(x,t_1)|$")
ax.plot(x, u_t2, linewidth=2.5, label=r"$|u(x,t_2)|$")
ax.plot(x, envelope, linewidth=2, linestyle="--", label=r"$Ce^{-a|x|^2}$ envelope")

ax.axvline(0, linewidth=1.2, alpha=0.55)
ax.annotate(
    "paired two-time\n(sister diagonal)\nconstraint input",
    xy=(0.35, 0.67),
    xytext=(2.05, 0.92),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=11,
    ha="left",
)

ax.set_title("Step 1: two-time decay assumption", fontsize=18, pad=14)
ax.set_xlabel("space coordinate x")
ax.set_ylabel("amplitude / bound")
ax.set_xlim(-5, 5)
ax.set_ylim(0, 1.25)
ax.legend(loc="upper right", frameon=True)

add_caption(
    ax,
    "AB: assume fast decay at two distinct times.  NOW/lift5: paired sister-diagonal constraint input.",
)

plt.tight_layout()
out = FIG_DIR / "step1_two_time_decay_code.png"
save(fig, out)
plt.show()

print(f"Saved: {out}")

## 3. Carleman-style exponential weighting

A Carleman step introduces a weight \(e^{\phi(x,t)}\), often with a large parameter.

Conceptual role:

\[
v(x,t) = e^{\phi(x,t)}u(x,t).
\]

For readability, this figure uses a bounded weighting lens. The paper’s formal section states the unbounded weighted-norm form. The figure is a finite-grid companion, not a theorem statement.

CGCS wording:

> **Same math · new scale.**  
> Hidden math must **extend (step)** and **resist (collapse)** under the constraints.


In [ ]:
weight = bounded_weight_lens(x, tau=0.22, floor=0.45, height=0.70)

weighted_t1 = weighted_signal(u_t1, weight)
weighted_t2 = weighted_signal(u_t2, weight)

weighted_t1_norm = normalize(weighted_t1)
weighted_t2_norm = normalize(weighted_t2)

print("Computed finite-grid weighted profiles.")
print("This is a visualization of scale change, not a formal Carleman estimate.")

## 4. Figure — Step 2: exponential / Carleman weight

Paper-ready filename:

```text
figures/step2_exponential_weight_code.png
```

Recommended caption:

> Step 2. Carleman weighting: same math, new scale. The weighted profiles make two-time smallness act under constraints.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2))

ax.plot(x, normalize(u_t1), linewidth=2.5, label=r"original $|u(x,t_1)|$ (normalized)")
ax.plot(x, weighted_t1_norm, linewidth=2.5, label=r"weighted $e^{\phi}|u(x,t_1)|$ (normalized)")
ax.plot(x, normalize(weight), linewidth=2, linestyle="--", label=r"weight profile (normalized)")

ax.annotate(
    "same math · new scale",
    xy=(0.45, weighted_t1_norm[np.argmin(np.abs(x - 0.45))]),
    xytext=(1.75, 0.90),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=12,
    ha="left",
)

ax.annotate(
    "smallness acts\nunder constraints",
    xy=(-0.8, weighted_t1_norm[np.argmin(np.abs(x + 0.8))]),
    xytext=(-3.9, 0.55),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=11,
    ha="left",
)

ax.set_title("Step 2: Exponential (Carleman) weighting", fontsize=18, pad=14)
ax.set_xlabel("space coordinate x")
ax.set_ylabel("normalized amplitude")
ax.set_xlim(-5, 5)
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", frameon=True)

add_caption(
    ax,
    "AB: introduce weighted norms.  NOW/lift5: hidden math extends (step) and resists (collapse) under constraints.",
)

plt.tight_layout()
out = FIG_DIR / "step2_exponential_weight_code.png"
save(fig, out)
plt.show()

print(f"Saved: {out}")

## 5. Local CGCS alignment for Notebook 01

This internal metric checks whether the explanation stays aligned with the standard proof roles:

| Step | AB role | NOW/lift5 role |
|---|---|---|
| two-time decay | assumption | sister diagonal constraint input |
| Carleman weight | weighted estimate | same math · new scale |
| continuity of language | proof structure preserved | extend (step), resist (collapse) |

A target of \(0.96\) marks max-CGCS clarity for this local notebook.


In [ ]:
steps = ["two-time decay", "Carleman weight", "language consistency"]
scores = np.array([0.985, 0.972, 0.965])

fig, ax = plot_alignment_bars(
    labels=steps,
    scores=scores,
    title="Notebook 01: local AB ↔ NOW alignment",
    target=0.96,
)

add_caption(ax, "All three local checks remain above the max-CGCS target.")

plt.tight_layout()
out = FIG_DIR / "notebook01_ab_now_alignment_code.png"
save(fig, out)
plt.show()

print(f"Saved: {out}")

## 6. Export summary

Generated files:

```text
figures/step1_two_time_decay_code.png
figures/step2_exponential_weight_code.png
figures/notebook01_ab_now_alignment_code.png
```

Next notebooks:

- `02_hardy_gate_demo.ipynb`
- `03_contradiction_pipeline_pde_aligned.ipynb`


In [ ]:
for file in [
    "step1_two_time_decay_code.png",
    "step2_exponential_weight_code.png",
    "notebook01_ab_now_alignment_code.png",
]:
    p = FIG_DIR / file
    print(f"{p}  exists={p.exists()}")